# 03 - Link interaction data to learning outcomes

This notebook runs the outcome analysis and presents the exact ID/session match, classroom-held-out predictions, treatment score contrast, and score-model SHAP summary. The analysis script is run by this notebook cell, and its current tables and figures are displayed below. The primary result is the **later unassisted exam** (`Part3Tot`). Assisted practice (`Part2Tot`) happened with AI available, so it is a secondary outcome. All score findings come from actual published scores; prompts or message counts are never treated as marks.

## 1. Confirm the earlier text-feature step is complete

Run notebook 01 first so the student-session features exist.

In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import display, Image

candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(
    (p for p in candidates if (p / "Research_Analysis" / "run_outcome_analysis.py").is_file()
     and (p / "outputs" / "research" / "datasets" / "student_session_features.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run notebooks/01_text_features_and_data.ipynb first, then reopen this notebook.")
OUT = PROJECT_ROOT / "outputs" / "research"
OUTCOME_OUT = OUT / "outcomes"
print(f"Project: {PROJECT_ROOT}")


Project: C:\Users\SrijanWork\Documents\Coding\Data-MIning-Project


## 2. Join the scores and run outcome models

The join uses exact student and session IDs. Predictive validation holds out whole classrooms, which prevents students or classrooms from leaking across train/test folds.

In [2]:
script = PROJECT_ROOT / "Research_Analysis" / "run_outcome_analysis.py"
run = subprocess.run(
    [sys.executable, str(script)],
    cwd=PROJECT_ROOT,
    check=True,
    text=True,
    capture_output=True,
)
print(run.stdout[-5000:])
if run.stderr.strip():
    print("\nScript messages (stderr):\n", run.stderr[-2000:])


566641748508622,
      "RMSE_sd": 0.016046364495079932,
      "R2_sd": 0.06320920877692354
    }
  ],
  "best_models_part3": {
    "baseline_context": {
      "model": "RandomForest",
      "RMSE_mean": 0.20405259901876022,
      "MAE_mean": 0.15877056084702063,
      "R2_mean": 0.378500600236343
    },
    "combined": {
      "model": "RandomForest",
      "RMSE_mean": 0.2052466267457978,
      "MAE_mean": 0.16264255037095604,
      "R2_mean": 0.3715533749118357
    },
    "existing_behavior": {
      "model": "RandomForest",
      "RMSE_mean": 0.20271741632373247,
      "MAE_mean": 0.1592574578964902,
      "R2_mean": 0.3868377778311974
    },
    "new_text": {
      "model": "RandomForest",
      "RMSE_mean": 0.2050171166838243,
      "MAE_mean": 0.16186962939488764,
      "R2_mean": 0.37285058405493954
    }
  },
  "incremental_comparisons": [
    {
      "comparison": "existing_behavior vs baseline (Ridge)",
      "model": "Ridge",
      "baseline_RMSE": 0.21969834110971337,
     

## 3. Check the exact score-match coverage

In [3]:
summary = json.loads((OUTCOME_OUT / "outcome_run_summary.json").read_text(encoding="utf-8"))
coverage_keys = [
    "matched_students", "classes", "total_outcome_rows_for_local_ai_students",
    "matched_student_sessions_with_chat", "outcome_rows_without_chat_for_session",
    "chat_sessions_without_score_row", "duplicate_score_session_keys",
]
display(pd.DataFrame([{"check": key, "value": summary.get(key)} for key in coverage_keys]))
print("Treatment mapping:", summary["treatment_mapping_validated"])


,check,value
0,matched_students,518
1,classes,27
2,total_outcome_rows_for_local_ai_students,1820
3,matched_student_sessions_with_chat,1735
4,outcome_rows_without_chat_for_session,85
5,chat_sessions_without_score_row,36
6,duplicate_score_session_keys,0


Treatment mapping: {'vanilla': 'GPT Base', 'aug': 'GPT Tutor'}


## 4. Compare the predictions

Lower RMSE and MAE indicate smaller prediction errors. The `new_text` feature set tests whether the added message clues help beyond context; we evaluate it on classrooms kept out of training, to check how well it works on unseen classes.

In [4]:
results = pd.read_csv(OUTCOME_OUT / "tables" / "outcome_grouped_cv_results.csv")
part3 = results.loc[results["outcome"].eq("Part3Tot")].sort_values("RMSE_mean")
display(part3[["feature_set", "model", "rows", "students", "classes", "MAE_mean", "RMSE_mean", "R2_mean"]].head(12))
ridge = part3.loc[part3["model"].eq("Ridge")]
display(ridge[["feature_set", "MAE_mean", "RMSE_mean", "R2_mean"]])


,feature_set,model,rows,students,classes,MAE_mean,RMSE_mean,R2_mean
8,existing_behavior,RandomForest,1735,518,27,0.159257,0.202717,0.386838
3,baseline_context,RandomForest,1735,518,27,0.158771,0.204053,0.378501
13,new_text,RandomForest,1735,518,27,0.161870,0.205017,0.372851
18,combined,RandomForest,1735,518,27,0.162643,0.205247,0.371553
4,baseline_context,GradientBoosting,1735,518,27,0.172895,0.212613,0.326076
9,existing_behavior,GradientBoosting,1735,518,27,0.173962,0.213765,0.318679
14,new_text,GradientBoosting,1735,518,27,0.174451,0.214324,0.315240
19,combined,GradientBoosting,1735,518,27,0.174782,0.214925,0.311261
12,new_text,Lasso,1735,518,27,0.178484,0.218939,0.285463
17,combined,Lasso,1735,518,27,0.178402,0.218975,0.285234


,feature_set,MAE_mean,RMSE_mean,R2_mean
1,baseline_context,0.177044,0.219327,0.281606
11,new_text,0.177421,0.219730,0.279423
16,combined,0.178607,0.222095,0.263625
6,existing_behavior,0.179235,0.222580,0.259905


In [5]:
increment = pd.read_csv(OUTCOME_OUT / "tables" / "exam_incremental_feature_value.csv")
contrasts = pd.read_csv(OUTCOME_OUT / "tables" / "treatment_arm_score_contrasts.csv")
interactions = pd.read_csv(OUTCOME_OUT / "tables" / "exam_text_treatment_interactions.csv")
display(increment)
display(contrasts)
display(interactions.sort_values("interaction_fdr_q_value").head(15))


,comparison,model,baseline_RMSE,augmented_RMSE,rmse_reduction,rmse_reduction_class_bootstrap_ci95_low,rmse_reduction_class_bootstrap_ci95_high,bootstrap_replicates,classes_resampled
0,existing_behavior vs baseline (Ridge),Ridge,0.219698,0.222933,-0.003235,-0.005389,-0.001199,2000,27
1,new_text vs baseline (Ridge),Ridge,0.219698,0.220094,-0.000396,-0.002523,0.001832,2000,27
2,combined vs baseline (Ridge),Ridge,0.219698,0.222444,-0.002746,-0.005417,0.000504,2000,27


,outcome,contrast,coefficient,cluster_robust_se,ci95_low,ci95_high,p_value,n_rows,n_students,n_classes,interpretation
0,Part3Tot,GPT Tutor (aug) - GPT Base (vanilla),0.021855,0.018802,-0.016793,0.060502,2.556469e-01,1702,508,27,Randomized class-arm contrast within the obser...
1,Part2Tot,GPT Tutor (aug) - GPT Base (vanilla),0.225854,0.027150,0.170046,0.281661,8.435953e-09,1702,508,27,Randomized class-arm contrast within the obser...


,feature,coefficient_vanilla_per_sd,coefficient_treatment_interaction,interaction_cluster_robust_se,interaction_ci95_low,interaction_ci95_high,interaction_p_value,rows,students,classes,interaction_fdr_q_value
0,text_approach_comparison_mean,0.002268,-0.019502,0.004422,-0.028592,-0.010412,0.000159,1702,508,27,0.005103
1,hint_request_present,-0.016447,0.024042,0.007176,0.009291,0.038792,0.002476,1702,508,27,0.039611
2,multiple_question_rate,-0.007891,0.010533,0.003324,0.003700,0.017365,0.003892,1702,508,27,0.041513
3,hint_request_rate,-0.020257,0.030904,0.010186,0.009965,0.051842,0.005420,1702,508,27,0.043362
4,average_word_count,-0.007642,0.031419,0.013216,0.004252,0.058586,0.025083,1702,508,27,0.160533
5,refinement_request_rate,-0.004340,-0.015832,0.009373,-0.035099,0.003434,0.103149,1702,508,27,0.285391
6,conceptual_question_rate,0.002397,-0.012155,0.008041,-0.028685,0.004374,0.142695,1702,508,27,0.285391
7,explanation_request_rate,0.007111,-0.014407,0.007952,-0.030754,0.001939,0.081596,1702,508,27,0.285391
8,text_lexical_diversity_mean,0.000024,-0.022113,0.012890,-0.048609,0.004382,0.098142,1702,508,27,0.285391
9,text_math_terminology_present_mean,0.003231,0.020335,0.011612,-0.003535,0.044204,0.091714,1702,508,27,0.285391


## 5. Review every outcome figure and the model explanation

The notebook run generates and displays all three outcome figures: held-out model comparison, score distributions by treatment, and SHAP summary for the combined exam model. SHAP describes which inputs the fitted score model used. It is not evidence that those inputs caused the score.

In [ ]:
figure_names = [
    "outcome_model_comparison.png",
    "exam_scores_by_treatment.png",
    "exam_combined_model_shap.png",
]
figure_paths = [OUTCOME_OUT / "figures" / name for name in figure_names]
missing_figures = [path.name for path in figure_paths if not path.is_file()]
if missing_figures:
    raise FileNotFoundError("Outcome analysis did not produce expected figures: " + ", ".join(missing_figures))

for path in figure_paths:
    print(path.name)
    display(Image(filename=str(path)))

shap_summary = pd.read_csv(OUTCOME_OUT / "tables" / "exam_combined_model_shap.csv")
display(shap_summary.head(15))

## Essential interpretation

Use the outcome tables above for the current result. A prompt is an observable behavior, not a direct test of understanding. The text-feature results are associations, and their predictive value is judged by held-out classrooms. The matched local sample contains GPT Base and GPT Tutor but no no-AI control arm, so this local analysis cannot compare either AI arm against no AI.


## Questions you may be asked

- **What counts as the learning outcome?** The main outcome is the later unassisted exam score, `Part3Tot`. `Part2Tot` is assisted practice and is secondary.
- **How did you join chat to scores?** By exact student ID and session ID. That gives 1,735 score sessions with same-session chat, covering all 518 students.
- **Why hold out classrooms?** Students in one classroom share the same assigned AI arm. Keeping whole classrooms out of training avoids testing on classmates of students already used to train the model.
- **What did the text add?** In the Ridge comparison, adding the new text clues did not improve exam prediction: RMSE changed from about 0.21970 to 0.22009. Lower is better, so this small change is not an improvement.
- **Does the project prove that a way of asking a question causes learning?** No. Text-feature results are associations. This matched transcript sample has GPT Base and GPT Tutor, but no no-AI control group.
